# Fine-tune PhoBERT-base-v2.1 -- Phan loai cam xuc tieng Viet (Nang cap end-to-end)

## Muc tieu
Fine-tune `vinai/phobert-base-v2` cho bai toan phan loai cam xuc 3 lop:
- **0 = neg** (Tieu cuc)
- **1 = pos** (Tich cuc)
- **2 = neu** (Trung tinh)

## Nang cap so voi phien ban goc
- Stratified K-Fold (k=5) + Model Ensembling (OOF bagging)
- Focal Loss + Label Smoothing thay the CrossEntropy thong thuong
- Class-weighted loss tu dong tinh tu phan bo nhan
- Layer-wise Learning Rate Decay theo chieu sau cua BERT
- Gradual Unfreezing: unfreeze dan tu classifier -> encoder tren -> encoder duoi
- Augmentation tieng Viet: synonym swap, random deletion, random swap
- Custom callback: BestModelMemoryCallback, EarlyStoppingByF1, GradualUnfreezing
- Khong luu checkpoint giua epoch (save_strategy=no)
- Chi luu model cuoi cung cua moi fold len Drive

## Dataset
- **File**: `/content/drive/MyDrive/data/data_v2.1/data_v2.1.csv`
- **Cot**: `Review`, `Label_number`, `Label_text`
- **Mapping**: 0=neg, 1=pos, 2=neu


In [ ]:
# =============================================================================
# Cell 2: Cai dat thu vien can thiet
# =============================================================================
# underthesea: xu ly ngon ngu tu nhien tieng Viet (word tokenize cho augmentation)
!pip install -q transformers==4.44.2 datasets accelerate scikit-learn seaborn matplotlib
!pip install -q underthesea  # Tuy chon: ho tro augmentation tieng Viet tot hon


In [ ]:
# =============================================================================
# Cell 3: Import thu vien va cau hinh moi truong
# (Tai su dung tu notebook goc: logic seed, GPU check, LABEL_MAP)
# =============================================================================
import os
import sys
import gc
import copy
import random
import warnings
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import softmax as scipy_softmax

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    TrainerControl,
    TrainerState,
    DataCollatorWithPadding,
    set_seed,
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# -- Dat seed de dam bao ket qua lap lai duoc (tai su dung tu notebook goc) --
SEED = 42

def seed_everything(seed):
    """Thiet lap seed cho tat ca thu vien de ket qua reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    set_seed(seed)

seed_everything(SEED)

# -- Kiem tra GPU --
if not torch.cuda.is_available():
    print('[CANH BAO] Khong tim thay GPU.')
    print('Vao Runtime -> Change runtime type -> chon GPU (T4).')
    sys.exit(1)

gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU    : {gpu_name} ({gpu_mem:.1f} GB)')
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')

device = torch.device('cuda')

# -- Cau hinh chung (tai su dung tu notebook goc, khong thay doi mapping nhan) --
MODEL_NAME  = 'vinai/phobert-base-v2'
NUM_LABELS  = 3
MAX_LENGTH  = 256
LABEL_MAP   = {0: 'neg', 1: 'pos', 2: 'neu'}   # Khong thay doi mapping nhan
LABEL2ID    = {v: k for k, v in LABEL_MAP.items()}
LABEL_NAMES = ['neg', 'pos', 'neu']

# -- Sieu tham so training --
N_FOLDS             = 5       # So fold cho Stratified K-Fold
NUM_EPOCHS          = 8       # So epoch toi da (early stopping se ket thuc som hon)
BATCH_SIZE          = 16      # Phu hop voi VRAM T4 (15 GB)
GRAD_ACCUM_STEPS    = 2       # Effective batch = 16 * 2 = 32
BASE_LR             = 2e-5    # Learning rate co so cho classifier / layer tren
LR_DECAY_FACTOR     = 0.9     # He so giam LR theo do sau layer (layer-wise decay)
WEIGHT_DECAY        = 0.01
WARMUP_RATIO        = 0.1     # 10% buoc dau warm up
FOCAL_GAMMA         = 2.0     # Gamma cho focal loss
LABEL_SMOOTHING_EPS = 0.1     # Epsilon label smoothing
AUG_PROB            = 0.35    # Xac suat augment moi mau
AUG_RATIO           = 0.5     # Ty le mau augment so voi tap train goc
EARLY_STOP_PATIENCE = 3       # So epoch cho early stopping

# Lich unfreeze: {epoch_bat_dau: so_layer_duoc_unfreeze_tu_tren} (-1 = tat ca)
UNFREEZE_SCHEDULE = {
    0: 0,    # Epoch 0: chi train classifier
    1: 3,    # Epoch 1: unfreeze them 3 layer tren (9, 10, 11)
    2: 6,    # Epoch 2: unfreeze them 6 layer tren (6..11)
    3: -1,   # Epoch 3+: unfreeze tat ca
}

print(f'Device : {device}')
print(f'N_FOLDS: {N_FOLDS} | NUM_EPOCHS: {NUM_EPOCHS} | BATCH: {BATCH_SIZE} x {GRAD_ACCUM_STEPS}')
print(f'BASE_LR: {BASE_LR} | LR_DECAY: {LR_DECAY_FACTOR} | FOCAL_GAMMA: {FOCAL_GAMMA}')


In [ ]:
# =============================================================================
# Cell 4: Mount Google Drive va load dataset
# (Tai su dung ham resolve_data_path() tu notebook goc)
# =============================================================================
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT   = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / 'DO_AN_1_main'
OUTPUT_DIR   = str(PROJECT_ROOT / 'phobert-v2.1')
DATA_FILENAME = 'data_v2.1.csv'

os.makedirs(OUTPUT_DIR, exist_ok=True)


def resolve_data_path() -> str:
    """Tu dong tim file du lieu trong Google Drive.

    (Tai su dung nguyen ham nay tu notebook goc.)
    Uu tien cac duong dan pho bien truoc, neu khong tim thay thi quet toan bo Drive.
    """
    candidate_paths = [
        DRIVE_ROOT / 'data' / 'data_v2.1' / DATA_FILENAME,
        DRIVE_ROOT / 'data_v2.1' / DATA_FILENAME,
        DRIVE_ROOT / 'DO_AN_1' / 'data' / 'data_v2.1' / DATA_FILENAME,
        DRIVE_ROOT / 'datasets' / 'data_v2.1' / DATA_FILENAME,
        PROJECT_ROOT / 'data' / 'data_v2.1' / DATA_FILENAME,
    ]
    for candidate in candidate_paths:
        if candidate.exists():
            return str(candidate)

    matches = sorted(DRIVE_ROOT.rglob(DATA_FILENAME))
    if matches:
        print(f'[THONG BAO] Tim thay file tai: {matches[0]}')
        return str(matches[0])

    raise FileNotFoundError(
        'Khong tim thay file data_v2.1.csv trong Google Drive. '
        'Hay kiem tra lai ten file hoac upload file vao MyDrive.'
    )


DATA_PATH = resolve_data_path()
print(f'Duong dan du lieu : {DATA_PATH}')
print(f'Duong dan luu model: {OUTPUT_DIR}')

df = pd.read_csv(DATA_PATH)
print(f'\nSo luong mau ban dau: {len(df)}')
print(f'Cac cot             : {list(df.columns)}')
df.head()


In [ ]:
# =============================================================================
# Cell 5: Kiem tra cau truc du lieu va lam sach
# (Tai su dung logic lam sach tu notebook goc, bo sung them kiem tra)
# =============================================================================
print('=== THONG TIN DU LIEU BAN DAU ===')
print(f'Kich thuoc         : {df.shape}')
print(f'Gia tri null       :\n{df.isnull().sum()}')
print(f'So mau trung lap   : {df.duplicated().sum()}')

# Loai bo null va trung lap
df = df.dropna(subset=['Review', 'Label_number'])
df = df.drop_duplicates(subset=['Review'], keep='first')

# Chuyen sang int va kiem tra nhan hop le
df['Label_number'] = df['Label_number'].astype(int)
valid_labels = {0, 1, 2}
invalid_mask = ~df['Label_number'].isin(valid_labels)
if invalid_mask.any():
    print(f'[CANH BAO] Loai bo {invalid_mask.sum()} dong co nhan khong hop le.')
    df = df[~invalid_mask]

# Lam sach text co ban (tai su dung tu notebook goc)
df['Review'] = df['Review'].astype(str).str.strip()
df = df[df['Review'].str.len() > 0]
df = df.reset_index(drop=True)

print(f'\n=== SAU KHI LAM SACH ===')
print(f'So luong mau: {len(df)}')
print(f'Phan bo nhan:')
label_counts = df['Label_number'].value_counts().sort_index()
for idx, count in label_counts.items():
    pct = count / len(df) * 100
    print(f'  {LABEL_MAP[idx]:>4}: {count:>6} mau ({pct:.1f}%)')


In [ ]:
# =============================================================================
# Cell 6: Phan tich du lieu (EDA)
# (Tai su dung logic EDA tu notebook goc)
# =============================================================================
colors = ['#e74c3c', '#2ecc71', '#3498db']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Bieu do 1: Phan bo nhan (bar chart)
label_counts = df['Label_number'].value_counts().sort_index()
bar_labels   = [LABEL_MAP[i] for i in label_counts.index]
bars = axes[0].bar(bar_labels, label_counts.values, color=colors, edgecolor='black')
axes[0].set_title('Phan bo nhan cam xuc', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Nhan')
axes[0].set_ylabel('So luong')
for bar, count in zip(bars, label_counts.values):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
        str(count), ha='center', fontsize=10, fontweight='bold'
    )

# Bieu do 2: Ti le nhan (pie chart)
axes[1].pie(
    label_counts.values, labels=bar_labels, colors=colors,
    autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11}
)
axes[1].set_title('Ti le nhan cam xuc', fontsize=14, fontweight='bold')

# Bieu do 3: Phan bo do dai cau (histogram)
df['word_count'] = df['Review'].apply(lambda x: len(x.split()))
axes[2].hist(df['word_count'], bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[2].axvline(df['word_count'].mean(), color='red', linestyle='--',
                label=f'Trung binh: {df["word_count"].mean():.1f}')
axes[2].axvline(df['word_count'].median(), color='green', linestyle='--',
                label=f'Trung vi: {df["word_count"].median():.1f}')
axes[2].set_title('Phan bo do dai cau (so tu)', fontsize=14, fontweight='bold')
axes[2].set_xlabel('So tu')
axes[2].set_ylabel('So luong')
axes[2].legend()

plt.tight_layout()
plt.show()

p95 = int(np.percentile(df['word_count'], 95))
print(f'Thong ke do dai cau: mean={df["word_count"].mean():.1f} | '
      f'median={df["word_count"].median():.1f} | '
      f'p95={p95} | max={df["word_count"].max()}')
print(f'MAX_LENGTH={MAX_LENGTH} bao phu p95: {"co" if MAX_LENGTH >= p95 else "khong"}')

df = df.drop(columns=['word_count'])


In [ ]:
# =============================================================================
# Cell 7: Augmentation tieng Viet
# Ky thuat: synonym swap, random deletion, random swap vi tri tu
# Back-translation: chi kich hoat neu googletrans kha dung
# =============================================================================

# -- Tu dien dong nghia don gian cho cac tu pho bien trong sentiment --
# Phu hop voi domain review san pham tieng Viet
SYNONYM_DICT = {
    'tot':        ['hay', 'dep', 'on', 'kha', 'duoc', 'chac chan'],
    'xau':        ['te', 'kem', 'do', 'toi', 'chan'],
    'dep':        ['tot', 'xinh', 'lung linh', 'dep mat', 'sang'],
    'te':         ['xau', 'kem', 'toi', 'do', 'chan'],
    'nhanh':      ['mau', 'le', 'tuc thi', 'nhanh chong'],
    'cham':       ['lau', 'cham chap', 'i ach', 'tre'],
    're':         ['gia phai chang', 'binh dan', 'hop tui tien', 'roi'],
    'dat':        ['mac', 'cao gia', 'dat do', 'khong roi'],
    'hai long':   ['thoa man', 'vua y', 'ung y', 'thoai mai'],
    'that vong':  ['buon', 'khong hai long', 'chan', 'khong ung'],
    'chat luong': ['chat', 'tot', 'on dinh', 'ben'],
    'san pham':   ['hang', 'mon', 'do', 'vat pham'],
    'mua':        ['dat', 'order', 'lay', 'chon'],
    'dung':       ['su dung', 'xai', 'trai nghiem'],
    'tuyet':      ['xuat sac', 'tuyet voi', 'tot lam', 'qua dep'],
    'binh thuong': ['tam duoc', 'on', 'trung binh', 'khong dac biet'],
    'giao hang':  ['ship', 'van chuyen', 'giao'],
    'dong goi':   ['bao bi', 'bao goi', 'dong'],
    'ho tro':     ['phuc vu', 'cham soc', 'tu van'],
    'hang':       ['san pham', 'do', 'mon', 'vat pham'],
    'gia':        ['chi phi', 'muc gia', 'gia ca'],
    'dep':        ['xinh dep', 'dep trai', 'sang trong'],
}


def synonym_swap(text: str, p: float = 0.15) -> str:
    """Thay the mot vai tu bang tu dong nghia trong dictionary.

    Dung phuong phap co ban (khong can underthesea) de dam bao on dinh.
    """
    words = text.split()
    new_words = []
    for word in words:
        w_lower = word.lower()
        if random.random() < p and w_lower in SYNONYM_DICT:
            new_words.append(random.choice(SYNONYM_DICT[w_lower]))
        else:
            new_words.append(word)
    return ' '.join(new_words)


def random_deletion(text: str, p: float = 0.10) -> str:
    """Xoa ngau nhien mot so tu, giu lai it nhat 3 tu."""
    words = text.split()
    if len(words) <= 3:
        return text
    new_words = [w for w in words if random.random() > p]
    return ' '.join(new_words) if new_words else random.choice(words)


def random_swap(text: str, n: int = 1) -> str:
    """Hoan doi ngau nhien vi tri cua 2 tu, lap n lan."""
    words = text.split()
    if len(words) < 2:
        return text
    words = words.copy()
    for _ in range(n):
        i, j = random.sample(range(len(words)), 2)
        words[i], words[j] = words[j], words[i]
    return ' '.join(words)


def augment_text(text: str) -> str:
    """Ap dung mot ky thuat augmentation ngau nhien."""
    choice = random.random()
    if choice < 0.45:
        return synonym_swap(text, p=0.15)
    elif choice < 0.75:
        return random_deletion(text, p=0.10)
    else:
        return random_swap(text, n=1)


def augment_dataset(
    texts: list,
    labels: list,
    aug_prob: float = 0.35,
    max_augmented: int = None,
) -> tuple:
    """Tao du lieu augmented cho tap train.

    Chi tao them mau moi (augmented samples), mau goc duoc giu nguyen o ngoai.
    Tra ve (augmented_texts, augmented_labels) de concat voi tap goc.

    Args:
        texts: Danh sach cac cau goc.
        labels: Danh sach nhan tuong ung.
        aug_prob: Xac suat augment moi mau.
        max_augmented: So mau augmented toi da (mac dinh = len(texts) * AUG_RATIO).

    Returns:
        (augmented_texts, augmented_labels): Chi cac mau duoc tao them.
    """
    if max_augmented is None:
        max_augmented = int(len(texts) * AUG_RATIO)

    indices = list(range(len(texts)))
    random.shuffle(indices)

    aug_texts, aug_labels = [], []
    for idx in indices:
        if len(aug_texts) >= max_augmented:
            break
        if random.random() < aug_prob:
            new_text = augment_text(texts[idx])
            if new_text.strip() and new_text != texts[idx]:
                aug_texts.append(new_text)
                aug_labels.append(labels[idx])

    return aug_texts, aug_labels


# -- Kiem tra thu augmentation --
sample = 'San pham rat tot, chat luong tuyet voi, giao hang nhanh'
print(f'Goc         : {sample}')
print(f'Synonym swap: {synonym_swap(sample, p=0.3)}')
print(f'Rand delete : {random_deletion(sample, p=0.2)}')
print(f'Rand swap   : {random_swap(sample, n=2)}')
print(f'augment_text: {augment_text(sample)}')


In [ ]:
# =============================================================================
# Cell 8: Tokenizer va cac ham tao dataset
# (Tai su dung tokenize_function tu notebook goc, bo sung create_hf_dataset)
# =============================================================================
print('Dang tai PhoBERT tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer: {type(tokenizer).__name__}')
print(f'Vocab size: {tokenizer.vocab_size}')


def tokenize_function(examples):
    """Tokenize van ban bang PhoBERT tokenizer.

    (Tai su dung nguyen ham nay tu notebook goc.)
    Khong pad o day -- DataCollatorWithPadding se pad dong theo batch.
    """
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=MAX_LENGTH,
    )


def create_hf_dataset(df_fold: pd.DataFrame) -> Dataset:
    """Tao HuggingFace Dataset tu DataFrame cua mot fold.

    Args:
        df_fold: DataFrame co cot 'Review' va 'Label_number'.

    Returns:
        Dataset da duoc tokenize, format PyTorch.
    """
    hf_ds = Dataset.from_pandas(
        df_fold[['Review', 'Label_number']].rename(
            columns={'Review': 'text', 'Label_number': 'label'}
        )
    )
    hf_ds = hf_ds.map(tokenize_function, batched=True, batch_size=1000)
    hf_ds.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
    return hf_ds


# -- Dynamic padding: pad theo cau dai nhat trong batch (tiet kiem VRAM) --
# (Tai su dung tu notebook goc, them pad_to_multiple_of=8 cho Tensor Core)
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding='longest',
    pad_to_multiple_of=8,
)


# -- Ham tinh metrics (tai su dung tu notebook goc, mo rong them per-class F1) --
def compute_metrics(eval_pred):
    """Tinh accuracy, precision, recall, F1 (macro va weighted)."""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, preds)

    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
        labels, preds, average='weighted', zero_division=0
    )

    return {
        'accuracy':          accuracy,
        'precision_macro':   prec_m,
        'recall_macro':      rec_m,
        'f1_macro':          f1_m,
        'precision_weighted': prec_w,
        'recall_weighted':   rec_w,
        'f1_weighted':       f1_w,
    }


# -- Kiem tra tokenization --
sample_text = df['Review'].iloc[0]
tok_out     = tokenizer(sample_text, truncation=True, max_length=MAX_LENGTH)
print(f'Vi du tokenization:')
print(f'  Text    : {sample_text[:80]}...')
print(f'  So token: {len(tok_out["input_ids"])}')


In [ ]:
# =============================================================================
# Cell 9: Ham loss: Focal Loss + Label Smoothing + Class Weights
# Thay the CrossEntropy thong thuong de tang hieu suat tren du lieu mat can bang
# =============================================================================

class FocalLossWithLabelSmoothing(nn.Module):
    """Focal Loss ket hop Label Smoothing va Class Weights.

    Focal Loss giam trong so cac mau de phan loai (pt cao),
    tap trung hoc tu cac mau kho (pt thap). Ket hop label smoothing
    giam overfitting vao cac nhan chinh xac.

    Cong thuc: FL(pt) = -alpha_t * (1 - pt)^gamma * log(pt_smooth)

    Args:
        num_classes: So luong lop phan loai.
        gamma: He so focal (khuyen nghi: 2.0).
        alpha: Class weights (tensor [num_classes]). None = khong dung.
        label_smoothing: Epsilon label smoothing (khuyen nghi: 0.1).
    """

    def __init__(
        self,
        num_classes: int,
        gamma: float = 2.0,
        alpha=None,
        label_smoothing: float = 0.1,
    ):
        super().__init__()
        self.num_classes    = num_classes
        self.gamma          = gamma
        self.label_smoothing = label_smoothing

        if alpha is not None:
            self.register_buffer('alpha', torch.tensor(alpha, dtype=torch.float))
        else:
            self.alpha = None

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # logits : [B, C]  |  targets: [B] (long)
        B, C = logits.shape
        eps  = self.label_smoothing

        # --- Label smoothing: phan phoi dich chuyen tu one-hot ---
        # P_smooth(y=c) = (1-eps) neu c==target, else eps/(C-1)
        with torch.no_grad():
            smooth_targets = torch.full_like(logits, eps / max(C - 1, 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1.0 - eps)

        # --- Log-softmax va xac suat ---
        log_probs = F.log_softmax(logits, dim=-1)   # [B, C]
        probs     = torch.exp(log_probs)             # [B, C]

        # --- Focal weight: (1 - p_t)^gamma ---
        # p_t: xac suat du doan cho nhan dung
        pt           = probs.gather(1, targets.unsqueeze(1)).squeeze(1)  # [B]
        focal_weight = (1.0 - pt) ** self.gamma                          # [B]

        # --- Cross entropy voi smooth labels ---
        ce_loss = -(smooth_targets * log_probs).sum(dim=-1)  # [B]

        # --- Ap dung focal weight ---
        loss = focal_weight * ce_loss  # [B]

        # --- Ap dung class weight (alpha) neu co ---
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            loss    = alpha_t * loss

        return loss.mean()


def compute_class_weights(labels: list) -> list:
    """Tinh class weights tu phan bo nhan trong tap train.

    Su dung sklearn compute_class_weight voi strategy='balanced'.
    Lop co it mau hon se duoc gan trong so lon hon.
    """
    classes = np.array([0, 1, 2])
    weights = compute_class_weight(
        class_weight='balanced',
        classes=classes,
        y=np.array(labels),
    )
    return weights.tolist()


# -- Kiem tra thu tren du lieu mau --
dummy_logits  = torch.randn(8, NUM_LABELS)
dummy_targets = torch.randint(0, NUM_LABELS, (8,))
sample_weights = compute_class_weights(df['Label_number'].tolist())
dummy_loss_fn = FocalLossWithLabelSmoothing(
    num_classes=NUM_LABELS,
    gamma=FOCAL_GAMMA,
    alpha=sample_weights,
    label_smoothing=LABEL_SMOOTHING_EPS,
)
dummy_loss = dummy_loss_fn(dummy_logits, dummy_targets)
print(f'Class weights (tu toan bo dataset): {[f"{w:.4f}" for w in sample_weights]}')
print(f'Focal Loss thu nghiem: {dummy_loss.item():.4f}')
print('FocalLossWithLabelSmoothing OK')


In [ ]:
# =============================================================================
# Cell 10: Layer-wise Learning Rate Decay Optimizer
# Cac layer sau (gan dau ra) nhan LR cao hon, layer dau (gan input) nhan LR thap hon
# Ky thuat nay giup fine-tune on dinh: tranh phu destroy features da hoc
# =============================================================================

def get_layerwise_optimizer(
    model,
    base_lr: float = 2e-5,
    decay_factor: float = 0.9,
    weight_decay: float = 0.01,
) -> AdamW:
    """Tao AdamW optimizer voi LR giam dan theo chieu sau cua BERT.

    Cau truc LR theo thu tu tu cao xuong thap:
      Classifier head  : base_lr
      Encoder layer 11 : base_lr * decay_factor^1
      Encoder layer 10 : base_lr * decay_factor^2
      ...
      Encoder layer 0  : base_lr * decay_factor^12
      Embeddings        : base_lr * decay_factor^13

    Args:
        model: Model PhoBERT voi classification head.
        base_lr: Learning rate co so (cho classifier).
        decay_factor: He so giam LR moi layer (0 < decay_factor < 1).
        weight_decay: Weight decay cho AdamW.

    Returns:
        AdamW optimizer voi parameter groups phan biet.
    """
    # Cac tham so khong ap dung weight decay (bias va LayerNorm)
    no_decay = ['bias', 'LayerNorm.weight']

    def make_group(params_list, lr, wd):
        """Tach params thanh 2 nhom: co va khong co weight decay."""
        with_decay    = [p for n, p in params_list if not any(nd in n for nd in no_decay)]
        without_decay = [p for n, p in params_list if     any(nd in n for nd in no_decay)]
        groups = []
        if with_decay:
            groups.append({'params': with_decay,    'lr': lr, 'weight_decay': wd})
        if without_decay:
            groups.append({'params': without_decay, 'lr': lr, 'weight_decay': 0.0})
        return groups

    optimizer_grouped_params = []

    # 1. Classifier head (LR cao nhat)
    cls_params = [(n, p) for n, p in model.named_parameters() if 'classifier' in n]
    optimizer_grouped_params += make_group(cls_params, base_lr, weight_decay)

    # 2. Cac transformer encoder layers (giam dan tu layer 11 xuong 0)
    for layer_depth, layer_idx in enumerate(range(11, -1, -1)):
        layer_lr     = base_lr * (decay_factor ** (layer_depth + 1))
        layer_prefix = f'roberta.encoder.layer.{layer_idx}.'
        layer_params = [
            (n, p) for n, p in model.named_parameters()
            if layer_prefix in n
        ]
        optimizer_grouped_params += make_group(layer_params, layer_lr, weight_decay)

    # 3. Embedding layers (LR thap nhat)
    emb_lr     = base_lr * (decay_factor ** 13)
    emb_params = [(n, p) for n, p in model.named_parameters() if 'roberta.embeddings' in n]
    optimizer_grouped_params += make_group(emb_params, emb_lr, weight_decay)

    optimizer = AdamW(
        optimizer_grouped_params,
        betas=(0.9, 0.999),
        eps=1e-8,
    )
    return optimizer


# -- In cau truc LR de kiem tra --
print('Cau truc Layer-wise LR Decay:')
print(f'  Classifier  : {BASE_LR:.2e}')
for i in range(11, 8, -1):
    depth = 11 - i + 1
    lr_i  = BASE_LR * (LR_DECAY_FACTOR ** depth)
    print(f'  Layer {i:>2}    : {lr_i:.2e}')
print(f'  ... (decay factor: {LR_DECAY_FACTOR})')
print(f'  Layer 0     : {BASE_LR * LR_DECAY_FACTOR**12:.2e}')
print(f'  Embeddings  : {BASE_LR * LR_DECAY_FACTOR**13:.2e}')


In [ ]:
# =============================================================================
# Cell 11: Custom Callbacks
# Ba callback: GradualUnfreezing, BestModelMemory, EarlyStoppingByF1
# =============================================================================

# ----- Callback 1: Gradual Unfreezing -----

class GradualUnfreezingCallback(TrainerCallback):
    """Unfreeze cac layer BERT dan dan theo epoch.

    Bat dau chi train classifier, sau do mo dan cac encoder layers tu tren xuong.
    Giup fine-tuning on dinh hon: tranh gradient shock lam hu features da hoc.

    Args:
        unfreeze_schedule: Dict {epoch_bat_dau: n_layers}.
            n_layers = 0  : chi train classifier.
            n_layers = k  : unfreeze k layers tren cung (11, 10, ..., 12-k).
            n_layers = -1 : unfreeze tat ca.
    """

    def __init__(self, unfreeze_schedule: dict = None):
        self.unfreeze_schedule = unfreeze_schedule or {}
        self._applied_epochs   = set()

    def _set_requires_grad(self, model, name_filter, value: bool):
        for n, p in model.named_parameters():
            if name_filter(n):
                p.requires_grad = value

    def _apply_schedule(self, model, n_layers: int, epoch: int):
        # Freeze tat ca truoc
        for p in model.parameters():
            p.requires_grad = False

        # Luon unfreeze classifier
        self._set_requires_grad(model, lambda n: 'classifier' in n, True)

        if n_layers == -1:
            # Unfreeze tat ca
            for p in model.parameters():
                p.requires_grad = True
            action_str = 'Unfreeze TẤT CA'
        elif n_layers > 0:
            # Unfreeze n layer tren cung
            for depth in range(n_layers):
                layer_idx = 11 - depth
                if layer_idx < 0:
                    break
                prefix = f'roberta.encoder.layer.{layer_idx}.'
                self._set_requires_grad(model, lambda n, p=prefix: p in n, True)
            action_str = f'Unfreeze top {n_layers} encoder layers + classifier'
        else:
            action_str = 'Chi train classifier'

        total     = sum(p.numel() for p in model.parameters())
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  [Gradual Unfreeze] Epoch {epoch}: {action_str}')
        print(f'    Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

    def on_train_begin(self, args, state, control, model=None, **kwargs):
        if 0 in self.unfreeze_schedule:
            self._apply_schedule(model, self.unfreeze_schedule[0], epoch=0)
            self._applied_epochs.add(0)
        else:
            self._apply_schedule(model, 0, epoch=0)

    def on_epoch_begin(self, args, state, control, model=None, **kwargs):
        epoch = int(round(state.epoch)) if state.epoch else 0
        if epoch in self._applied_epochs:
            return
        # Tim schedule phu hop nhat (epoch lon nhat <= current epoch)
        matching = [e for e in self.unfreeze_schedule if e <= epoch]
        if not matching:
            return
        target_epoch = max(matching)
        if target_epoch not in self._applied_epochs:
            self._apply_schedule(model, self.unfreeze_schedule[target_epoch], epoch=epoch)
            self._applied_epochs.add(target_epoch)


# ----- Callback 2: Best Model Memory (thay cho load_best_model_at_end) -----

class BestModelMemoryCallback(TrainerCallback):
    """Luu state_dict cua model tot nhat vao RAM thay vi disk.

    Cho phep save_strategy='no' trong khi van phuc hoi duoc best model sau training.
    Best model duoc chon theo metric (mac dinh: eval_f1_macro, cao hon = tot hon).
    """

    def __init__(self, metric: str = 'eval_f1_macro', greater_is_better: bool = True):
        self.metric            = metric
        self.greater_is_better = greater_is_better
        self.best_score        = None
        self.best_state_dict   = None
        self.best_epoch        = None

    def _is_better(self, current: float) -> bool:
        if self.best_score is None:
            return True
        return (current > self.best_score) if self.greater_is_better \
               else (current < self.best_score)

    def on_evaluate(self, args, state, control, metrics=None, model=None, **kwargs):
        if metrics is None or self.metric not in metrics:
            return
        current = metrics[self.metric]
        if self._is_better(current):
            self.best_score  = current
            self.best_epoch  = state.epoch
            # Sao chep vao CPU de giai phong VRAM
            self.best_state_dict = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            print(f'\n  [Best Model] Cap nhat: {self.metric}={current:.4f} (epoch {state.epoch:.1f})')

    def load_best_model(self, model) -> None:
        """Tai best model weights vao model hien tai."""
        if self.best_state_dict is None:
            print('  [Best Model] Khong co best state, giu nguyen model hien tai.')
            return
        model.load_state_dict(self.best_state_dict)
        model.to(device)
        print(f'  [Best Model] Da tai: {self.metric}={self.best_score:.4f} (epoch {self.best_epoch:.1f})')


# ----- Callback 3: Early Stopping theo F1 -----

class EarlyStoppingByF1Callback(TrainerCallback):
    """Dung training som neu eval_f1_macro khong cai thien trong 'patience' epoch.

    Khac voi EarlyStoppingCallback mac dinh cua HuggingFace (can save_strategy != no),
    callback nay chi can ghi nhan metric va set control.should_training_stop.

    Args:
        patience: So epoch cho phep khong cai thien truoc khi dung.
        min_delta: Nguong cai thien toi thieu de coi la 'co tien bo'.
    """

    def __init__(self, patience: int = 3, min_delta: float = 0.001):
        self.patience   = patience
        self.min_delta  = min_delta
        self.best_f1    = None
        self.wait_count = 0

    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        current_f1 = (metrics or {}).get('eval_f1_macro', 0.0)
        if self.best_f1 is None or current_f1 > self.best_f1 + self.min_delta:
            self.best_f1    = current_f1
            self.wait_count = 0
        else:
            self.wait_count += 1
            remaining = self.patience - self.wait_count
            print(f'  [Early Stop] F1={current_f1:.4f}, best={self.best_f1:.4f} '
                  f'(khong cai thien {self.wait_count}/{self.patience})')
            if self.wait_count >= self.patience:
                control.should_training_stop = True
                print(f'  [Early Stop] Dung training sau {self.patience} epoch khong cai thien F1.')


print('Cac callback da dinh nghia: GradualUnfreezingCallback, '
      'BestModelMemoryCallback, EarlyStoppingByF1Callback')


In [ ]:
# =============================================================================
# Cell 12: Custom Trainer (PhoBERTTrainer)
# Override compute_loss de dung FocalLoss + LabelSmoothing
# Override create_optimizer de dung layerwise LR decay
# =============================================================================

class PhoBERTTrainer(Trainer):
    """Trainer tuy chinh cho PhoBERT sentiment classification.

    Nang cap tu Trainer goc:
    1. compute_loss: Focal Loss + Label Smoothing + Class Weights
    2. create_optimizer: AdamW voi layer-wise LR decay
    """

    def __init__(self, *args, loss_fn: nn.Module = None, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_fn = loss_fn

    def compute_loss(self, model, inputs, return_outputs: bool = False, **kwargs):
        """Tinh loss bang FocalLossWithLabelSmoothing thay vi CrossEntropy."""
        labels  = inputs.pop('labels')
        outputs = model(**inputs)
        logits  = outputs.logits

        if self.loss_fn is not None:
            loss = self.loss_fn(logits, labels)
        else:
            # Fallback ve CrossEntropy neu khong co loss_fn
            loss = F.cross_entropy(logits, labels)

        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self):
        """Tao AdamW voi layer-wise LR decay."""
        if self.optimizer is None:
            self.optimizer = get_layerwise_optimizer(
                self.model,
                base_lr=self.args.learning_rate,
                decay_factor=LR_DECAY_FACTOR,
                weight_decay=self.args.weight_decay,
            )
        return self.optimizer


print('PhoBERTTrainer da dinh nghia.')
print('  - compute_loss: FocalLoss + LabelSmoothing + ClassWeights')
print('  - create_optimizer: AdamW voi layer-wise LR decay')


In [ ]:
# =============================================================================
# Cell 13: Stratified K-Fold Training
# Moi fold: augment -> tokenize -> train voi all nang cap -> luu model cuoi
# Ket qua OOF duoc gom lai de tinh ensemble performance
# =============================================================================

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Ma tran luu xac suat OOF (out-of-fold) de ensemble sau nay
oof_probs  = np.zeros((len(df), NUM_LABELS), dtype=np.float32)
oof_labels = df['Label_number'].values.copy()

fold_metrics_list = []
fold_log_histories = []

print(f'=== BAT DAU STRATIFIED {N_FOLDS}-FOLD TRAINING ===')
print(f'Kich thuoc dataset: {len(df)} mau')
print(f'Kich thuoc moi fold val: ~{len(df)//N_FOLDS} mau')
print()

for fold, (train_idx, val_idx) in enumerate(
    skf.split(df['Review'].values, df['Label_number'].values)
):
    print(f'\n{'='*60}')
    print(f'FOLD {fold + 1}/{N_FOLDS}')
    print(f'{'='*60}')

    # -- Chia du lieu --
    train_fold_df = df.iloc[train_idx].reset_index(drop=True)
    val_fold_df   = df.iloc[val_idx].reset_index(drop=True)

    print(f'Train: {len(train_fold_df)} mau | Val: {len(val_fold_df)} mau')
    print(f'Phan bo train: {dict(train_fold_df["Label_number"].value_counts().sort_index())}')

    # -- Augmentation cho tap train cua fold nay --
    seed_everything(SEED + fold)  # Seed khac nhau moi fold
    aug_texts, aug_labels = augment_dataset(
        texts         = train_fold_df['Review'].tolist(),
        labels        = train_fold_df['Label_number'].tolist(),
        aug_prob      = AUG_PROB,
        max_augmented = int(len(train_fold_df) * AUG_RATIO),
    )
    print(f'Augmentation: +{len(aug_texts)} mau')

    # Gop du lieu goc va augmented
    aug_df = pd.DataFrame({'Review': aug_texts, 'Label_number': aug_labels})
    combined_train_df = pd.concat(
        [train_fold_df[['Review', 'Label_number']], aug_df],
        ignore_index=True,
    )
    print(f'Tap train sau augment: {len(combined_train_df)} mau')

    # -- Tokenize --
    print('Dang tokenize...')
    train_dataset = create_hf_dataset(combined_train_df)
    val_dataset   = create_hf_dataset(val_fold_df)

    # -- Khoi tao model moi cho moi fold (fresh from pretrained) --
    seed_everything(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=NUM_LABELS,
        id2label=LABEL_MAP,
        label2id=LABEL2ID,
    )
    model.to(device)

    # -- Tinh class weights tu tap train cua fold (loai tru augmented) --
    fold_class_weights = compute_class_weights(
        train_fold_df['Label_number'].tolist()
    )
    print(f'Class weights fold {fold+1}: {[f"{w:.3f}" for w in fold_class_weights]}')

    # -- Khoi tao loss function voi class weights cua fold --
    fold_loss_fn = FocalLossWithLabelSmoothing(
        num_classes    = NUM_LABELS,
        gamma          = FOCAL_GAMMA,
        alpha          = fold_class_weights,
        label_smoothing = LABEL_SMOOTHING_EPS,
    ).to(device)

    # -- Tinh so buoc de cau hinh scheduler + warmup --
    effective_batch  = BATCH_SIZE * GRAD_ACCUM_STEPS
    steps_per_epoch  = max(len(train_dataset) // effective_batch, 1)
    total_steps      = steps_per_epoch * NUM_EPOCHS
    warmup_steps     = int(total_steps * WARMUP_RATIO)

    # -- Training Arguments (save_strategy='no': khong luu checkpoint giua epoch) --
    fold_output_dir = f'/tmp/phobert_fold_{fold+1}'
    training_args = TrainingArguments(
        output_dir               = fold_output_dir,
        overwrite_output_dir     = True,

        # So epoch va batch
        num_train_epochs         = NUM_EPOCHS,
        per_device_train_batch_size = BATCH_SIZE,
        per_device_eval_batch_size  = BATCH_SIZE * 2,
        gradient_accumulation_steps = GRAD_ACCUM_STEPS,

        # Learning rate va optimizer
        learning_rate            = BASE_LR,
        weight_decay             = WEIGHT_DECAY,
        adam_beta1               = 0.9,
        adam_beta2               = 0.999,
        adam_epsilon             = 1e-8,
        max_grad_norm            = 1.0,   # Gradient clipping

        # Scheduler voi warmup
        lr_scheduler_type        = 'cosine',
        warmup_steps             = warmup_steps,

        # Mixed precision fp16 (T4 ho tro)
        fp16                     = True,

        # Evaluation theo epoch
        eval_strategy            = 'epoch',
        logging_strategy         = 'steps',
        logging_steps            = 50,

        # Khong luu checkpoint giua cac epoch
        save_strategy            = 'no',

        # Logging
        report_to                = 'none',
        disable_tqdm             = False,

        # Seed va toi uu memory
        seed                     = SEED,
        data_seed                = SEED,
        dataloader_num_workers   = 2,
        dataloader_pin_memory    = True,
        group_by_length          = True,  # Nhom cau cung do dai -> giam padding
    )

    # -- Khoi tao callbacks --
    best_model_cb  = BestModelMemoryCallback(
        metric='eval_f1_macro', greater_is_better=True
    )
    early_stop_cb  = EarlyStoppingByF1Callback(
        patience=EARLY_STOP_PATIENCE, min_delta=0.001
    )
    unfreeze_cb    = GradualUnfreezingCallback(
        unfreeze_schedule=UNFREEZE_SCHEDULE
    )

    # -- Khoi tao PhoBERTTrainer --
    trainer = PhoBERTTrainer(
        model         = model,
        args          = training_args,
        train_dataset = train_dataset,
        eval_dataset  = val_dataset,
        tokenizer     = tokenizer,
        data_collator = data_collator,
        compute_metrics = compute_metrics,
        loss_fn       = fold_loss_fn,
        callbacks     = [best_model_cb, early_stop_cb, unfreeze_cb],
    )

    # -- Giai phong VRAM truoc khi train --
    torch.cuda.empty_cache()
    gc.collect()

    # -- Train --
    print(f'\nBat dau training fold {fold+1}...')
    t0 = time.time()
    train_result = trainer.train()
    elapsed = time.time() - t0
    print(f'Training fold {fold+1} hoan tat trong {elapsed/60:.1f} phut')

    # -- Tai best model tu RAM --
    best_model_cb.load_best_model(model)

    # -- Predict tren val fold de lay OOF probs --
    pred_out = trainer.predict(val_dataset)
    fold_probs = scipy_softmax(pred_out.predictions, axis=-1).astype(np.float32)
    oof_probs[val_idx] = fold_probs

    # -- Tinh metrics cho fold nay --
    fold_preds = np.argmax(fold_probs, axis=-1)
    fold_true  = val_fold_df['Label_number'].values

    fold_acc  = accuracy_score(fold_true, fold_preds)
    _, _, fold_f1_macro, _ = precision_recall_fscore_support(
        fold_true, fold_preds, average='macro', zero_division=0
    )
    _, _, fold_f1_weighted, _ = precision_recall_fscore_support(
        fold_true, fold_preds, average='weighted', zero_division=0
    )
    fold_metrics_list.append({
        'fold': fold + 1,
        'accuracy':    fold_acc,
        'f1_macro':    fold_f1_macro,
        'f1_weighted': fold_f1_weighted,
        'train_loss':  train_result.training_loss,
    })
    print(f'\nFold {fold+1} OOF: Accuracy={fold_acc:.4f} | '
          f'F1_macro={fold_f1_macro:.4f} | F1_weighted={fold_f1_weighted:.4f}')

    # Luu log history de ve bieu do sau
    fold_log_histories.append(list(trainer.state.log_history))

    # -- Luu model cuoi cung cua fold len Drive --
    fold_save_dir = os.path.join(OUTPUT_DIR, f'fold_{fold+1}')
    os.makedirs(fold_save_dir, exist_ok=True)
    model.save_pretrained(fold_save_dir)
    tokenizer.save_pretrained(fold_save_dir)

    # Luu fold metrics kem theo
    fold_meta = {
        **fold_metrics_list[-1],
        'model_name': MODEL_NAME,
        'label_map': LABEL_MAP,
    }
    with open(os.path.join(fold_save_dir, 'fold_metrics.json'), 'w', encoding='utf-8') as f:
        json.dump(fold_meta, f, ensure_ascii=False, indent=2)
    print(f'Da luu model fold {fold+1} vao: {fold_save_dir}')

    # -- Giai phong memory --
    del model, trainer, fold_loss_fn
    del train_dataset, val_dataset
    del best_model_cb, early_stop_cb, unfreeze_cb
    torch.cuda.empty_cache()
    gc.collect()

print(f'\n{'='*60}')
print('HOAN TAT K-FOLD TRAINING')
print(f'{'='*60}')


In [ ]:
# =============================================================================
# Cell 14: Ensemble OOF Evaluation - Danh gia hieu suat cuoi cung
# Trung binh xac suat tu K model tren cac val fold tuong ung
# =============================================================================

print('=== KET QUA TUNG FOLD ===')
print(f'{"Fold":>5} {"Accuracy":>10} {"F1 Macro":>10} {"F1 Weighted":>12} {"Train Loss":>12}')
print('-' * 55)
for m in fold_metrics_list:
    print(f'{m["fold"]:>5} {m["accuracy"]:>10.4f} {m["f1_macro"]:>10.4f} '
          f'{m["f1_weighted"]:>12.4f} {m["train_loss"]:>12.4f}')

# Tinh trung binh qua cac fold
avg_acc  = np.mean([m['accuracy']    for m in fold_metrics_list])
avg_f1m  = np.mean([m['f1_macro']    for m in fold_metrics_list])
avg_f1w  = np.mean([m['f1_weighted'] for m in fold_metrics_list])
std_f1m  = np.std( [m['f1_macro']    for m in fold_metrics_list])
print('-' * 55)
print(f'{"Mean":>5} {avg_acc:>10.4f} {avg_f1m:>10.4f} {avg_f1w:>12.4f}')
print(f'{"Std":>5} {"":>10} {std_f1m:>10.4f}')

print(f'\n=== ENSEMBLE OOF EVALUATION ===')
# Ensemble: trung binh xac suat OOF tu tat ca cac fold
ensemble_preds = np.argmax(oof_probs, axis=-1)

ens_acc  = accuracy_score(oof_labels, ensemble_preds)
prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
    oof_labels, ensemble_preds, average='macro', zero_division=0
)
prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
    oof_labels, ensemble_preds, average='weighted', zero_division=0
)

print(f'Ensemble Accuracy           : {ens_acc:.4f}')
print(f'Ensemble Precision (macro)  : {prec_m:.4f}')
print(f'Ensemble Recall (macro)     : {rec_m:.4f}')
print(f'Ensemble F1-score (macro)   : {f1_m:.4f}')
print(f'Ensemble F1-score (weighted): {f1_w:.4f}')

print(f'\n=== CLASSIFICATION REPORT (ENSEMBLE OOF) ===')
print(classification_report(oof_labels, ensemble_preds, target_names=LABEL_NAMES, digits=4))

# Luu ensemble metrics tong hop
ensemble_metrics = {
    'ensemble_accuracy':    ens_acc,
    'ensemble_f1_macro':    f1_m,
    'ensemble_f1_weighted': f1_w,
    'ensemble_precision_macro': prec_m,
    'ensemble_recall_macro': rec_m,
    'per_fold': fold_metrics_list,
    'mean_f1_macro': avg_f1m,
    'std_f1_macro':  std_f1m,
    'model_name': MODEL_NAME,
    'n_folds': N_FOLDS,
    'label_map': LABEL_MAP,
    'config': {
        'max_length': MAX_LENGTH, 'batch_size': BATCH_SIZE,
        'grad_accum': GRAD_ACCUM_STEPS, 'base_lr': BASE_LR,
        'lr_decay': LR_DECAY_FACTOR, 'focal_gamma': FOCAL_GAMMA,
        'label_smoothing': LABEL_SMOOTHING_EPS, 'aug_prob': AUG_PROB,
    }
}


In [ ]:
# =============================================================================
# Cell 15: Visualization -- Training curves, Confusion Matrix, Fold Comparison
# (Mo rong tu notebook goc: them so sanh fold va ensemble confusion matrix)
# =============================================================================

fig = plt.figure(figsize=(22, 14))

# ---- Hang 1: Training loss va Eval F1 cho tung fold ----
ax1 = fig.add_subplot(2, 3, 1)
ax2 = fig.add_subplot(2, 3, 2)

colors_fold = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']

for fold_idx, log_hist in enumerate(fold_log_histories):
    # Training loss
    steps  = [e.get('step', 0)  for e in log_hist if 'loss' in e and 'eval_loss' not in e]
    losses = [e['loss']          for e in log_hist if 'loss' in e and 'eval_loss' not in e]
    if steps:
        ax1.plot(steps, losses, alpha=0.7, linewidth=1.2,
                 color=colors_fold[fold_idx], label=f'Fold {fold_idx+1}')

    # Eval F1
    epochs = [e.get('epoch', 0)       for e in log_hist if 'eval_f1_macro' in e]
    f1s    = [e['eval_f1_macro']       for e in log_hist if 'eval_f1_macro' in e]
    if epochs:
        ax2.plot(epochs, f1s, 'o-', alpha=0.8, linewidth=1.5,
                 color=colors_fold[fold_idx], label=f'Fold {fold_idx+1}')

ax1.set_title('Training Loss (tung fold)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')
ax1.legend(fontsize=8)
ax1.grid(True, alpha=0.3)

ax2.set_title('Eval F1 Macro (tung fold)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('F1 Macro')
ax2.set_ylim([0.5, 1.05])
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

# ---- Hang 1, cot 3: So sanh F1 macro giua cac fold ----
ax3 = fig.add_subplot(2, 3, 3)
fold_names = [f'Fold {m["fold"]}' for m in fold_metrics_list]
fold_f1s   = [m['f1_macro'] for m in fold_metrics_list]
bar_colors = [colors_fold[i] for i in range(N_FOLDS)]
bars = ax3.bar(fold_names, fold_f1s, color=bar_colors, edgecolor='black')
ax3.axhline(avg_f1m, color='black', linestyle='--', linewidth=1.5, label=f'Mean={avg_f1m:.4f}')
for bar, val in zip(bars, fold_f1s):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{val:.4f}', ha='center', fontsize=9)
ax3.set_title('F1 Macro moi fold', fontsize=13, fontweight='bold')
ax3.set_ylabel('F1 Macro')
ax3.set_ylim([max(0, min(fold_f1s) - 0.05), 1.0])
ax3.legend(fontsize=9)
ax3.grid(True, alpha=0.3, axis='y')

# ---- Hang 2: Confusion Matrix OOF ----
ax4 = fig.add_subplot(2, 3, 4)
cm  = confusion_matrix(oof_labels, ensemble_preds)
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
    ax=ax4, cbar_kws={'shrink': 0.8},
    annot_kws={'fontsize': 12, 'fontweight': 'bold'},
)
ax4.set_title(f'Confusion Matrix (Ensemble OOF)\nAccuracy={ens_acc:.4f}',
             fontsize=13, fontweight='bold')
ax4.set_xlabel('Du doan')
ax4.set_ylabel('Thuc te')

# ---- Hang 2, cot 2: Precision/Recall/F1 per class ----
ax5 = fig.add_subplot(2, 3, 5)
prec_pc, rec_pc, f1_pc, _ = precision_recall_fscore_support(
    oof_labels, ensemble_preds, average=None, zero_division=0
)
x      = np.arange(NUM_LABELS)
width  = 0.25
ax5.bar(x - width, prec_pc, width, label='Precision', color='#3498db', alpha=0.8)
ax5.bar(x,         rec_pc,  width, label='Recall',    color='#2ecc71', alpha=0.8)
ax5.bar(x + width, f1_pc,   width, label='F1',        color='#e74c3c', alpha=0.8)
ax5.set_xticks(x)
ax5.set_xticklabels(LABEL_NAMES)
ax5.set_title('Precision / Recall / F1 theo lop', fontsize=13, fontweight='bold')
ax5.set_ylabel('Score')
ax5.set_ylim([0.0, 1.1])
ax5.legend(fontsize=9)
ax5.grid(True, alpha=0.3, axis='y')
for i, (p, r, f) in enumerate(zip(prec_pc, rec_pc, f1_pc)):
    ax5.text(i - width, p + 0.01, f'{p:.3f}', ha='center', fontsize=7)
    ax5.text(i,         r + 0.01, f'{r:.3f}', ha='center', fontsize=7)
    ax5.text(i + width, f + 0.01, f'{f:.3f}', ha='center', fontsize=7)

# ---- Hang 2, cot 3: OOF prob distribution ----
ax6 = fig.add_subplot(2, 3, 6)
max_conf = np.max(oof_probs, axis=-1)
correct  = (ensemble_preds == oof_labels)
ax6.hist(max_conf[correct],  bins=40, alpha=0.6, color='#2ecc71', label='Dung')
ax6.hist(max_conf[~correct], bins=40, alpha=0.6, color='#e74c3c', label='Sai')
ax6.set_title('Phan bo do tin cay du doan', fontsize=13, fontweight='bold')
ax6.set_xlabel('Xac suat max (do tin cay)')
ax6.set_ylabel('So luong mau')
ax6.legend(fontsize=9)
ax6.grid(True, alpha=0.3)

plt.suptitle(
    f'PhoBERT {N_FOLDS}-Fold Ensemble | '
    f'OOF Accuracy={ens_acc:.4f} | OOF F1_macro={f1_m:.4f}',
    fontsize=15, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()

# In confusion matrix dang so
print('\n=== CONFUSION MATRIX (Ensemble OOF) ===')
print(f'{"":>8} {"neg":>8} {"pos":>8} {"neu":>8}')
for i, name in enumerate(LABEL_NAMES):
    row = '  '.join(f'{cm[i][j]:>6}' for j in range(NUM_LABELS))
    print(f'{name:>8} {row}')


In [ ]:
# =============================================================================
# Cell 16: Luu tokenizer, ensemble config va metrics len Google Drive
# Cac model da duoc luu vao fold_1..fold_k trong Cell 13
# =============================================================================

# Luu tokenizer vao thu muc goc (dung chung cho tat ca fold)
tokenizer_dir = os.path.join(OUTPUT_DIR, 'tokenizer')
os.makedirs(tokenizer_dir, exist_ok=True)
tokenizer.save_pretrained(tokenizer_dir)
print(f'Tokenizer da luu vao: {tokenizer_dir}')

# Luu ensemble metrics tong hop
ensemble_metrics_path = os.path.join(OUTPUT_DIR, 'ensemble_metrics.json')
with open(ensemble_metrics_path, 'w', encoding='utf-8') as f:
    json.dump(ensemble_metrics, f, ensure_ascii=False, indent=2)
print(f'Ensemble metrics da luu vao: {ensemble_metrics_path}')

# Luu ensemble config de su dung khi inference
ensemble_config = {
    'model_name': MODEL_NAME,
    'n_folds':    N_FOLDS,
    'num_labels': NUM_LABELS,
    'max_length': MAX_LENGTH,
    'label_map':  LABEL_MAP,
    'label2id':   LABEL2ID,
    'fold_dirs':  [f'fold_{i+1}' for i in range(N_FOLDS)],
    'tokenizer_dir': 'tokenizer',
    'oof_f1_macro':  f1_m,
    'oof_accuracy':  ens_acc,
}
config_path = os.path.join(OUTPUT_DIR, 'ensemble_config.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(ensemble_config, f, ensure_ascii=False, indent=2)
print(f'Ensemble config da luu vao: {config_path}')

# Kiem tra cac file da luu
print(f'\nTong ket cac file da luu vao {OUTPUT_DIR}:')
for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, '').count(os.sep)
    indent = '  ' * level
    folder_name = os.path.basename(root)
    if level == 0:
        print(f'{indent}[{folder_name}/]')
    else:
        print(f'{indent}[{folder_name}/]')
    subindent = '  ' * (level + 1)
    for fname in sorted(files):
        fpath   = os.path.join(root, fname)
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f'{subindent}{fname} ({size_mb:.1f} MB)')

torch.cuda.empty_cache()
print(f'\nHoan tat luu tru. GPU memory: {torch.cuda.memory_allocated(0)/1e9:.2f} GB used')


In [ ]:
# =============================================================================
# Cell 17: Inference Demo -- Du doan cam xuc voi Ensemble
# Tai tat ca K model, trung binh xac suat, tra ve ket qua ensemble
# (Mo rong ham predict_sentiment tu notebook goc sang ensemble inference)
# =============================================================================

def load_ensemble_models(output_dir: str, n_folds: int, device):
    """Tai tat ca K model tu cac thu muc fold tuong ung.

    Args:
        output_dir: Thu muc goc chua cac thu muc fold_1..fold_k.
        n_folds: So fold (so model can tai).
        device: Device de tai model len.

    Returns:
        (models_list, tokenizer): Danh sach K model va tokenizer.
    """
    loaded_tokenizer = AutoTokenizer.from_pretrained(
        os.path.join(output_dir, 'tokenizer')
    )
    models = []
    for k in range(1, n_folds + 1):
        fold_dir = os.path.join(output_dir, f'fold_{k}')
        m = AutoModelForSequenceClassification.from_pretrained(fold_dir)
        m.to(device)
        m.eval()
        models.append(m)
        print(f'  Da tai model fold {k} tu: {fold_dir}')
    return models, loaded_tokenizer


def predict_sentiment_ensemble(
    text: str,
    models_list: list,
    tok,
    device,
    max_len: int = MAX_LENGTH,
) -> dict:
    """Du doan cam xuc bang ensemble cua K model.

    Trung binh xac suat tu tat ca K model truoc khi chon nhan.
    Cho ket qua on dinh va chinh xac hon du doan don le.

    (Mo rong ham predict_sentiment tu notebook goc.)

    Args:
        text: Cau tieng Viet can phan tich cam xuc.
        models_list: Danh sach K model da fine-tune.
        tok: PhoBERT tokenizer.
        device: Device de chay inference.
        max_len: Do dai toi da cua dau vao.

    Returns:
        Dict chua label, label_id, confidence, probabilities, per_model_probs.
    """
    inputs = tok(
        text,
        return_tensors='pt',
        truncation=True,
        max_length=max_len,
        padding=True,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    all_probs = []
    with torch.no_grad():
        for m in models_list:
            logits = m(**inputs).logits
            probs  = torch.softmax(logits, dim=-1).cpu().numpy()[0]
            all_probs.append(probs)

    # Trung binh xac suat cua K model
    avg_probs = np.mean(all_probs, axis=0)

    pred_id   = int(np.argmax(avg_probs))
    pred_name = LABEL_MAP[pred_id]
    confidence = float(avg_probs[pred_id])

    return {
        'text':           text,
        'label':          pred_name,
        'label_id':       pred_id,
        'confidence':     confidence,
        'probabilities':  {LABEL_MAP[i]: float(avg_probs[i]) for i in range(NUM_LABELS)},
        'n_models':       len(models_list),
    }


# -- Tai tat ca K model --
print('Dang tai ensemble models...')
ensemble_models, ensemble_tokenizer = load_ensemble_models(OUTPUT_DIR, N_FOLDS, device)
print(f'Da tai {len(ensemble_models)} model thanh cong.\n')

# -- Inference demo voi cac cau mau --
# (Tai su dung tap mau tu notebook goc, bo sung them truong hop trung tinh)
test_sentences = [
    'San pham rat tot, chat luong tuyet voi, se mua lai',
    'Hang kem chat luong, giao sai mau, that vong',
    'Hang binh thuong, khong co gi dac biet',
    'Doi ngu ho tro nhiet tinh, giao hang nhanh, dong goi can than',
    'Mua ve dung 2 ngay da hong, tien nao cua nay',
    'Mau sac dep, size vua vat, gia hop ly',
    'Dich vu cham soc khach hang kha tot, se quay lai',
    'Hang gui thieu, phai doi qua lau, khong hai long',
]

print('=== INFERENCE DEMO (Ensemble K-Fold)\n')
for sentence in test_sentences:
    result = predict_sentiment_ensemble(
        sentence, ensemble_models, ensemble_tokenizer, device
    )
    prob_str = ' | '.join(
        f'{name}: {prob:.4f}' for name, prob in result['probabilities'].items()
    )
    print(f'Cau    : {result["text"]}')
    print(f'Ket qua: {result["label"]} (do tin cay: {result["confidence"]:.4f})')
    print(f'Chi tiet: {prob_str}')
    print('-' * 70)

# -- Giai phong VRAM sau inference --
for m in ensemble_models:
    m.cpu()
del ensemble_models
torch.cuda.empty_cache()
print(f'\nHoan tat. GPU memory: {torch.cuda.memory_allocated(0)/1e9:.2f} GB used')
print(f'Tat ca model da duoc luu tai: {OUTPUT_DIR}')
